# 05 · Integración supervisado + no supervisado

Carga artefactos de `03` y `04`, cruza predicciones supervisadas con clusters,
tópicos, similitudes y metadata literaria.

Este notebook no entrena modelos y no recalcula features.

In [1]:
from __future__ import annotations

import sys

# Permite importar modules/ y utils/ cuando el notebook se ejecuta desde notebook/
sys.path.insert(0, "..")

%load_ext autoreload
%autoreload 2

# python
import json

import pandas as pd

# modules
from modules.integration import (
    parse_json_list,
    json_dumps_list,
    build_integrated_results,
    cluster_tag_crosstab,
    cluster_topic_crosstab,
    cluster_poet_crosstab,
    cluster_source_crosstab,
    poet_tag_crosstab,
    top_tags_by_cluster,
    build_vallejo_view,
)
from modules.io import (
    artifact_path,
    data_path,
    load_csv,
    save_csv,
    save_json,
)

# typings
from pandas import DataFrame as PandasDF
from typing import Dict

# setup
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("max_colwidth", None)

## 1. Cargar artefactos

Entradas esperadas:

- `data/poems_processed.csv`
- `artifacts/unsupervised/unsupervised_results.csv`
- `artifacts/supervised/supervised_predictions.csv`

In [2]:
poems_df: PandasDF = load_csv(
    data_path("poems_processed.csv")
)

unsupervised_df: PandasDF = load_csv(
    artifact_path("unsupervised", "unsupervised_results.csv")
)

supervised_pred_df: PandasDF = load_csv(
    artifact_path("supervised", "supervised_predictions.csv")
)

if "tags" in poems_df.columns:
    poems_df["tags"] = poems_df["tags"].apply(parse_json_list)

if "predicted_tags_json" in supervised_pred_df.columns:
    supervised_pred_df["predicted_tags"] = supervised_pred_df[
        "predicted_tags_json"
    ].apply(parse_json_list)

print("poems_df:", poems_df.shape)
print("unsupervised_df:", unsupervised_df.shape)
print("supervised_pred_df:", supervised_pred_df.shape)

display(poems_df.head(2))
display(unsupervised_df.head(3))
display(supervised_pred_df.head(3))

poems_df: (13858, 13)
unsupervised_df: (13751, 18)
supervised_pred_df: (13751, 9)


,poem_id,source_row_id,poem_hash,title,title_raw,poet,poet_raw,poem,tags,source,corpus_role,poem_raw,poem_processed
0,poetry_foundation::000000::d1e86366cdbb,0,d1e86366cdbb,objects used to prop open a window,\r\r\n Objects Used to Prop Open a Window\r\r\n,michelle menting,Michelle Menting,dog bone stapler cribbage board garlic press because this window is looselacks suction lacks grip bungee cord bootstrap dog leash leather belt because this window had sash cords they frayed they broke feather duster thatch of straw empty bottle of elmers glue because this window is loudits hinges clack open clack shut stuffed bear baby blanket single crib newel because this window is split its dividing in two velvet moss sagebrush willow branch robins wing because this window its paneless its only a frame of air,[],poetry_foundation,reference,"\r\r\nDog bone, stapler,\r\r\ncribbage board, garlic press\r\r\n because this window is loose—lacks\r\r\nsuction, lacks grip.\r\r\nBungee cord, bootstrap,\r\r\ndog leash, leather belt\r\r\n because this window had sash cords.\r\r\nThey frayed. They broke.\r\r\nFeather duster, thatch of straw, empty\r\r\nbottle of Elmer's glue\r\r\n because this window is loud—its hinges clack\r\r\nopen, clack shut.\r\r\nStuffed bear, baby blanket,\r\r\nsingle crib newel\r\r\n because this window is split. It's dividing\r\r\nin two.\r\r\nVelvet moss, sagebrush,\r\r\nwillow branch, robin's wing\r\r\n because this window, it's pane-less. It's only\r\r\na frame of air.\r\r\n",dog bone stapler cribbag board garlic press window looselack suction lack grip bunge cord bootstrap dog leash leather belt window sash cord fray break feather duster thatch straw bottl elmer glue window loudit he clack open clack shut stuf bear babi blanket singl crib newel window split divid velvet moss sagebrush willow branch robin wing window paneless frame air
1,poetry_foundation::000001::0dbfeaec7a94,1,0dbfeaec7a94,the new church,\r\r\n The New Church\r\r\n,lucia cherciu,Lucia Cherciu,the old cupola glinted above the clouds shone among fir trees but it took him an hour for the half mile all the way up the hill as he trailed the village passed him by greeted him asked about his health but everybody hurried to catch the mass left him leaning against fences measuring the road with the walking stick he sculpted he yearned for the day when the new church would be builtright across the road now it rises above the moon saints in frescoes meet the eye and only the rain has started to cut through the shingles on the roof of his empty house the apple trees have taken over the sky sequestered the gate sidled over the porch,[],poetry_foundation,reference,"\r\r\nThe old cupola glinted above the clouds, shone\r\r\namong fir trees, but it took him an hour\r\r\nfor the half mile all the way up the hill. As he trailed,\r\r\nthe village passed him by, greeted him,\r\r\nasked about his health, but everybody hurried\r\r\nto catch the mass, left him leaning against fences,\r\r\nmeasuring the road with the walking stick he sculpted.\r\r\nHe yearned for the day when the new church\r\r\nwould be built—right across the road. Now\r\r\nit rises above the moon: saints in frescoes\r\r\nmeet the eye, and only the rain has started to cut\r\r\nthrough the shingles on the roof of his empty\r\r\nhouse. The apple trees have taken over the sky,\r\r\nsequestered the gate, sidled over the porch.\r\r\n",old cupola glint cloud shone fir tree take hour half mile way hill trail villag pass greet ask health everybodi hurri catch mass leave lean fenc measur road walk stick sculpt yearn day new church builtright road rise moon saint fresco meet eye rain start cut shingl roof hous appl tree take sky sequest gate sidl porch


,poem_id,title,title_raw,poet,poet_raw,source,corpus_role,cluster_km,cluster_gmm,cluster_agg,cluster_dbscan,lda_topic_id,lda_topic_prob,lda_topic_terms,nearest_titles_cosine_json,nearest_scores_cosine_json,embedding_x,embedding_y
0,poetry_foundation::000000::d1e86366cdbb,objects used to prop open a window,\r\r\n Objects Used to Prop Open a Window\r\r\n,michelle menting,Michelle Menting,poetry_foundation,reference,0,1,0,-1,1,0.6896,"like, tree, white, water, blue","[""confession of a bird watcher"", ""mother and child body and soul"", ""supple cord"", ""schizotableau"", ""winter""]","[0.1981, 0.1817, 0.1762, 0.1747, 0.173]",-25.025835,-55.625130
1,poetry_foundation::000001::0dbfeaec7a94,the new church,\r\r\n The New Church\r\r\n,lucia cherciu,Lucia Cherciu,poetry_foundation,reference,0,0,0,-1,1,0.6734,"like, tree, white, water, blue","[""uppity"", ""drifting at midday"", ""what about this"", ""photo of a man on sunset drive"", ""perseus in arkansas""]","[0.2142, 0.2016, 0.1951, 0.1939, 0.1926]",-39.418160,-14.665663
2,poetry_foundation::000002::52eddc66be9e,look for me,\r\r\n Look for Me\r\r\n,ted kooser,Ted Kooser,poetry_foundation,reference,0,1,0,-1,1,0.5274,"like, tree, white, water, blue","[""this is the time of grasshoppers and all that i see is dying"", ""ode to a grasshopper"", ""radiator"", ""pink slip at tool dye"", ""moths""]","[0.177, 0.1704, 0.1601, 0.1521, 0.147]",11.992420,32.972317


,poem_id,title,title_raw,poet,poet_raw,source,corpus_role,predicted_tags,predicted_tags_json
0,poetry_foundation::000000::d1e86366cdbb,objects used to prop open a window,\r\r\n Objects Used to Prop Open a Window\r\r\n,michelle menting,Michelle Menting,poetry_foundation,reference,[],[]
1,poetry_foundation::000001::0dbfeaec7a94,the new church,\r\r\n The New Church\r\r\n,lucia cherciu,Lucia Cherciu,poetry_foundation,reference,[nature],"[""nature""]"
2,poetry_foundation::000002::52eddc66be9e,look for me,\r\r\n Look for Me\r\r\n,ted kooser,Ted Kooser,poetry_foundation,reference,"[living, nature]","[""living"", ""nature""]"


## 2. Construir resultados integrados

La unión se hace por:

- `title`
- `poet`
- `source`
- `corpus_role`

Esto evita depender del orden de filas entre artefactos.

In [3]:
df_integracion: PandasDF = build_integrated_results(
    base_df=poems_df,
    unsupervised_df=unsupervised_df,
    supervised_predictions_df=supervised_pred_df,
    key_cols=["poem_id"],
)

display(df_integracion.head(10))
print(df_integracion.shape)

,poem_id,title,title_raw,poet,poet_raw,source,corpus_role,poem,poem_raw,poem_processed,original_tags,cluster_km,cluster_gmm,cluster_agg,cluster_dbscan,lda_topic_id,lda_topic_prob,lda_topic_terms,nearest_titles_cosine_json,nearest_scores_cosine_json,embedding_x,embedding_y,nearest_titles_cosine,nearest_scores_cosine,predicted_tags,predicted_tags_json
0,poetry_foundation::000000::d1e86366cdbb,objects used to prop open a window,\r\r\n Objects Used to Prop Open a Window\r\r\n,michelle menting,Michelle Menting,poetry_foundation,reference,dog bone stapler cribbage board garlic press because this window is looselacks suction lacks grip bungee cord bootstrap dog leash leather belt because this window had sash cords they frayed they broke feather duster thatch of straw empty bottle of elmers glue because this window is loudits hinges clack open clack shut stuffed bear baby blanket single crib newel because this window is split its dividing in two velvet moss sagebrush willow branch robins wing because this window its paneless its only a frame of air,"\r\r\nDog bone, stapler,\r\r\ncribbage board, garlic press\r\r\n because this window is loose—lacks\r\r\nsuction, lacks grip.\r\r\nBungee cord, bootstrap,\r\r\ndog leash, leather belt\r\r\n because this window had sash cords.\r\r\nThey frayed. They broke.\r\r\nFeather duster, thatch of straw, empty\r\r\nbottle of Elmer's glue\r\r\n because this window is loud—its hinges clack\r\r\nopen, clack shut.\r\r\nStuffed bear, baby blanket,\r\r\nsingle crib newel\r\r\n because this window is split. It's dividing\r\r\nin two.\r\r\nVelvet moss, sagebrush,\r\r\nwillow branch, robin's wing\r\r\n because this window, it's pane-less. It's only\r\r\na frame of air.\r\r\n",dog bone stapler cribbag board garlic press window looselack suction lack grip bunge cord bootstrap dog leash leather belt window sash cord fray break feather duster thatch straw bottl elmer glue window loudit he clack open clack shut stuf bear babi blanket singl crib newel window split divid velvet moss sagebrush willow branch robin wing window paneless frame air,[],0.0,1.0,0.0,-1.0,1.0,0.6896,"like, tree, white, water, blue","[""confession of a bird watcher"", ""mother and child body and soul"", ""supple cord"", ""schizotableau"", ""winter""]","[0.1981, 0.1817, 0.1762, 0.1747, 0.173]",-25.025835,-55.625130,"[confession of a bird watcher, mother and child body and soul, supple cord, schizotableau, winter]","[0.1981, 0.1817, 0.1762, 0.1747, 0.173]",[],[]
1,poetry_foundation::000001::0dbfeaec7a94,the new church,\r\r\n The New Church\r\r\n,lucia cherciu,Lucia Cherciu,poetry_foundation,reference,the old cupola glinted above the clouds shone among fir trees but it took him an hour for the half mile all the way up the hill as he trailed the village passed him by greeted him asked about his health but everybody hurried to catch the mass left him leaning against fences measuring the road with the walking stick he sculpted he yearned for the day when the new church would be builtright across the road now it rises above the moon saints in frescoes meet the eye and only the rain has started to cut through the shingles on the roof of his empty house the apple trees have taken over the sky sequestered the gate sidled over the porch,"\r\r\nThe old cupola glinted above the clouds, shone\r\r\namong fir trees, but it took him an hour\r\r\nfor the half mile all the way up the hill. As he trailed,\r\r\nthe village passed him by, greeted him,\r\r\nasked about his health, but everybody hurried\r\r\nto catch the mass, left him leaning against fences,\r\r\nmeasuring the road with the walking stick he sculpted.\r\r\nHe yearned for the day when the new church\r\r\nwould be built—right across the road. Now\r\r\nit rises above the moon: saints in frescoes\r\r\nmeet the eye, and only the rain has started to cut\r\r\nthrough the shingles on the roof of his empty\r\r\nhouse. The apple trees have taken over the sky,\r\r\nsequestered the gate, sidled over the porch.\r\

(13858, 26)


In [4]:
required_result_cols = [
    "cluster_km",
    "cluster_gmm",
    "cluster_agg",
    "cluster_dbscan",
    "lda_topic_id",
    "lda_topic_terms",
    "predicted_tags",
    "nearest_titles_cosine",
    "nearest_scores_cosine",
]

missing_result_cols = [
    col for col in required_result_cols
    if col not in df_integracion.columns
]

if missing_result_cols:
    print("Advertencia: faltan columnas esperadas:", missing_result_cols)
else:
    print("Columnas principales de integración disponibles.")

Columnas principales de integración disponibles.


## 3. Cruces de análisis

In [5]:
cluster_tag_matrix = cluster_tag_crosstab(
    df_integracion,
    cluster_col="cluster_km",
)

cluster_topic_matrix = cluster_topic_crosstab(
    df_integracion,
    cluster_col="cluster_km",
)

cluster_poet_matrix = cluster_poet_crosstab(
    df_integracion,
    cluster_col="cluster_km",
)

cluster_source_matrix = cluster_source_crosstab(
    df_integracion,
    cluster_col="cluster_km",
)

poet_tag_matrix = poet_tag_crosstab(
    df_integracion,
)

top_tags_cluster = top_tags_by_cluster(
    df_integracion,
    cluster_col="cluster_km",
    top_n=5,
)

vallejo_view = build_vallejo_view(
    df_integracion,
)

display(cluster_tag_matrix)
display(cluster_topic_matrix)
display(cluster_poet_matrix)
display(cluster_source_matrix)
display(poet_tag_matrix.head(20))
display(top_tags_cluster)
display(vallejo_view)

predicted_tags,activities,animals,arts sciences,christianity,cities urban life,death,desire,eating drinking,faith doubt,family ancestors,friends enemies,gender sexuality,god the divine,heartache loss,heavens,history politics,home life,infatuation crushes,jobs working,landscapes pastorals,language linguistics,life choices,living,love,marriage companionship,men women,money economics,music,mythology folklore,nature,painting sculpture,parenthood,pets,philosophy,planets,poetry poets,race ethnicity,reading books,realistic complicated,relationships,religion,rivers,romantic love,seas,social commentaries,sorrow grieving,stars,streams,summer,the body,the mind,time brevity,travels journeys,trees flowers,unrequited love,war conflict,weather,winter
cluster_km,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0.0,0.011,0.005,0.063,0.000,0.003,0.016,0.000,0.0,0.000,0.017,0.0,0.0,0.003,0.000,0.0,0.009,0.0,0.000,0.0,0.003,0.0,0.0,0.333,0.035,0.0,0.0,0.0,0.001,0.000,0.151,0.0,0.001,0.0,0.0,0.0,0.011,0.002,0.001,0.001,0.090,0.017,0.005,0.001,0.005,0.185,0.000,0.0,0.005,0.0,0.006,0.0,0.004,0.001,0.003,0.000,0.007,0.0,0.001
1.0,0.000,0.000,0.047,0.038,0.000,0.024,0.002,0.0,0.004,0.000,0.0,0.0,0.035,0.005,0.0,0.005,0.0,0.009,0.0,0.011,0.0,0.0,0.122,0.220,0.0,0.0,0.0,0.000,0.009,0.122,0.0,0.000,0.0,0.0,0.0,0.009,0.000,0.000,0.033,0.113,0.105,0.002,0.047,0.002,0.029,0.002,0.0,0.002,0.0,0.000,0.0,0.000,0.000,0.002,0.002,0.000,0.0,0.000


lda_topic_id,0.0,1.0,2.0,3.0,4.0,5.0,6.0
cluster_km,,,,,,,
0.0,0.036,0.176,0.097,0.058,0.263,0.243,0.127
1.0,0.005,0.011,0.164,0.803,0.003,0.005,0.008


poet        a b spellman  a e housman  a e stallings  a f moritz  a m juster  \
cluster_km                                                                     
0.0                  0.0        0.001          0.002       0.001         0.0   
1.0                  0.0        0.000          0.000       0.000         0.0   

poet        a r ammons  a v christie  aaron poochigian  aaron shurin  \
cluster_km                                                             
0.0              0.001           0.0               0.0           0.0   
1.0              0.000           0.0               0.0           0.0   

poet        aase berg  ab jackson  abid b alabras  abraham cowley  \
cluster_km                                                          
0.0               0.0         0.0             0.0             0.0   
1.0               0.0         0.0             0.0             0.0   

poet        abraham lincoln  abraham sutzkever  ada limon  adah isaacs menken  \
cluster_km                                                                      
0.0                     0.0                0.0      0.001                 0.0   
1.0                     0.0                0.0      0.000                 0.0   

poet        adam clay  adam fitzgerald  adam kirsch  adam zagajewski  \
cluster_km                                                             
0.0               0.0              0.0          0.0            0.001   
1.0               0.0              0.0          0.0            0.000   

poet        adelaide crapsey  adonis  adrian blevins  adrian c louis  \
cluster_km                                                             
0.0                    0.001   0.001             0.0             0.0   
1.0                    0.000   0.000             0.0             0.0   

poet        adrian castro  adrian koesters  adrian matejka  \
cluster_km                                                   
0.0                   0.0              0.0           0.001   
1.0                   0.0              0.0           0.000   

poet        adrien stoutenburg  adrienne rich  adrienne su  aemilia lanyer  \
cluster_km                                                                   
0.0                        0.0            0.0          0.0           0.000   
1.0                        0.0            0.0          0.0           0.003   

poet        afaa michael weaver  afshan shafi  agha shahid ali  \
cluster_km                                                       
0.0                       0.001           0.0            0.001   
1.0                       0.000           0.0            0.000   

poet        aharon shabtai  ahren warner     ai  aifric mac aodha  \
cluster_km                                                          
0.0                    0.0           0.0  0.001               0.0   
1.0                    0.0           0.0  0.000               0.0   

poet        ailbhe darcy  ailbhe ni ghearbhuigh  aileen lucia fisher  \
cluster_km                                                             
0.0                  0.0                    0.0                  0.0   
1.0                  0.0                    0.0                  0.0   

poet        ailish hopper  aimee nezhukumatathil  aja couchois duncan  \
cluster_km                                                              
0.0                   0.0                  0.001                  0.0   
1.0                   0.0                  0.000                  0.0   

poet        al maginnes  al young  alan dugan  alan feldman  alan felsenthal  \
cluster_km                                                                     
0.0                 0.0       0.0       0.001           0.0              0.0   
1.0                 0.0       0.0       0.000           0.0              0.0   

poet        alan gillis  alan pelaez lopez  alan r shapiro  alan seeger  \
cluster_km                                                                
0.0                 0.0                0.0           

source,cesar_vallejo,poetry_foundation
cluster_km,,
0.0,0.0,1.0
1.0,0.0,1.0


predicted_tags,activities,animals,arts sciences,christianity,cities urban life,death,desire,eating drinking,faith doubt,family ancestors,friends enemies,gender sexuality,god the divine,heartache loss,heavens,history politics,home life,infatuation crushes,jobs working,landscapes pastorals,language linguistics,life choices,living,love,marriage companionship,men women,money economics,music,mythology folklore,nature,painting sculpture,parenthood,pets,philosophy,planets,poetry poets,race ethnicity,reading books,realistic complicated,relationships,religion,rivers,romantic love,seas,social commentaries,sorrow grieving,stars,streams,summer,the body,the mind,time brevity,travels journeys,trees flowers,unrequited love,war conflict,weather,winter
poet,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
a e housman,0.0,0.000,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.348,0.087,0.0,0.0,0.0,0.000,0.0,0.087,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.087,0.043,0.0,0.0,0.0,0.217,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.043,0.0,0.087,0.0,0.0
a e stallings,0.0,0.042,0.125,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.458,0.042,0.0,0.0,0.0,0.000,0.0,0.208,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.125,0.000,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.0
a f moritz,0.0,0.000,0.000,0.0,0.0,0.036,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.500,0.000,0.0,0.0,0.0,0.000,0.0,0.143,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.143,0.000,0.0,0.0,0.0,0.143,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.036,0.0,0.000,0.0,0.0
a m juster,0.0,0.000,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,1.000,0.000,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.0
a r ammons,0.0,0.000,0.105,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.158,0.000,0.0,0.0,0.0,0.000,0.0,0.526,0.0,0.0,0.0,0.0,0.0,0.053,0.0,0.0,0.0,0.000,0.053,0.0,0.0,0.0,0.105,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.0
a v christie,0.0,0.000,0.333,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333,0.000,0.0,0.0,0.0,0.000,0.0,0.333,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.0
aaron shurin,0.0,0.000,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.500,0.000,0.0,0.0,0.0,0.000,0.0,0.500,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.0
aase berg,0.0,0.000,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333,0.000,0.0,0.0,0.0,0.000,0.0,0.167,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.333,0.0,0.0,0.0,0.0,0.167,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.0
ab jackson,0.0,0.000,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.0,1.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.0


,cluster_km,predicted_tags,count
0,0.0,living,6091
1,0.0,social commentaries,3385
2,0.0,nature,2767
3,0.0,relationships,1643
4,0.0,arts sciences,1148
5,1.0,love,121
6,1.0,living,67
7,1.0,nature,67
8,1.0,relationships,62
9,1.0,religion,58


,title,title_raw,poet,poet_raw,source,corpus_role,cluster_km,cluster_gmm,cluster_agg,cluster_dbscan,lda_topic_id,lda_topic_prob,lda_topic_terms,predicted_tags,nearest_titles_cosine,nearest_scores_cosine,embedding_x,embedding_y
0,homage vallejo,\r\r\n Homage: Vallejo\r\r\n,ishion hutchinson,Ishion Hutchinson,poetry_foundation,reference,0.0,1.0,0.0,102.0,1.0,0.7407,"like, tree, white, water, blue",[social commentaries],"[thirteen ways of looking at a blackbird, black stone on a white stone, ghazal after ferguson, black stone on top of a white stone, sloweddown blackbird]","[0.3723, 0.3086, 0.2982, 0.2948, 0.2852]",-17.932055,5.242527
1,after vallejo,\r\r\n After Vallejo\r\r\n,tom sleigh,Tom Sleigh,poetry_foundation,reference,0.0,1.0,0.0,-1.0,5.0,0.4070,"like, light, bodi, hand, eye",[living],"[east of carthage an idyll, being serious, how long, the ballad of reading gaol, song of myself version]","[0.2323, 0.232, 0.2276, 0.2242, 0.2241]",-20.786057,17.201680
2,variations on a text by vallejo,\r\r\n Variations on a Text by Vallejo\r\r\n,donald justice,Donald Justice,poetry_foundation,reference,0.0,0.0,0.0,-1.0,1.0,0.4737,"like, tree, white, water, blue","[living, relationships]","[a poem called day, animalistic hymn, forecast, a blessing for wedding, kef]","[0.2571, 0.249, 0.2441, 0.2217, 0.216]",14.576059,40.260690
3,black stone on a white stone,\r\r\n Black Stone on a White Stone\r\r\n,cesar vallejo,César Vallejo,poetry_foundation,reference,0.0,1.0,0.0,-1.0,6.0,0.9767,"man, like, say, know, love",[living],"[black stone on top of a white stone, identification, c, one thursday afternoon magdalena sonora, homage vallejo]","[0.7771, 0.4026, 0.3442, 0.31, 0.3086]",32.526573,-15.033323
4,miguel,\r\r\n Miguel\r\r\n,cesar vallejo,César Vallejo,poetry_foundation,reference,0.0,1.0,0.0,-1.0,6.0,0.4238,"man, like, say, know, love","[living, relationships]","[indian chant, jim trueblood father of the year, he laughed with a laugh, one train may hide another, this is a shout out for the tenants of the red little building on ocean avenue]","[0.2909, 0.2631, 0.2492, 0.2327, 0.2136]",-61.309002,-22.454210
5,the black heralds,The Black Heralds,cesar vallejo,César Vallejo,cesar_vallejo,external,0.0,0.0,0.0,-1.0,5.0,0.3044,"like, light, bodi, hand, eye","[living, social commentaries]","[the aim was song, ballad of the salvation army, keumganggul diamond cave, fortuna, which way the winds blow]","[0.225, 0.2077, 0.2053, 0.203, 0.2009]",29.504080,20.228811
6,black stone on top of a white stone,Black Stone On Top Of A White Stone,cesar vallejo,César Vallejo,cesar_vallejo,external,0.0,0.0,0.0,-1.0,6.0,0.9196,"man, like, say, know, love","[death, living]","[black stone on a white stone, identification, c, one thursday afternoon magdalena sonora, homage vallejo]","[0.7771, 0.3718, 0.3288, 0.3029, 0.2948]",38.994530,36.218890
7,paris october poem,"Paris, October 1936 Poem",cesar vallejo,César Vallejo,cesar_vallejo,external,0.0,0.0,0.0,-1.0,5.0,0.6257,"like, light, bodi, hand, eye",[],"[october, a kind of villanelle, the shirt, exodus, my blue shirt]","[0.2192, 0.1951, 0.1897, 0.1843, 0.184]",-21.742662,-35.655870
8,xiii,XIII,cesar vallejo,César Vallejo,cesar_vallejo,external,0.0,0.0,0.0,158.0,3.0,0.3149,"thi, thou, love, thee, shall",[living],"[more juice please, the noname tapestries, notes for the cactus poem, youd think the sky would run out of water, life story]","[0.2678, 0.266, 0.2419, 0.2352, 0.2327]",4.794347,36.070835


## 4. Metadata de features

Se guarda un resumen del pipeline y las dimensiones generadas.

In [6]:
integration_metadata: Dict = {
    "base_artifact": "data/poems_processed.csv",
    "unsupervised_artifact": "artifacts/unsupervised/unsupervised_results.csv",
    "supervised_artifact": "artifacts/supervised/supervised_predictions.csv",
    "integration_key_cols": [
        "title",
        "poet",
        "source",
        "corpus_role",
    ],
    "n_rows_integrated": int(df_integracion.shape[0]),
    "n_columns_integrated": int(df_integracion.shape[1]),
    "n_clusters_km": (
        int(df_integracion["cluster_km"].nunique())
        if "cluster_km" in df_integracion.columns
        else None
    ),
    "n_topics_lda": (
        int(df_integracion["lda_topic_id"].nunique())
        if "lda_topic_id" in df_integracion.columns
        else None
    ),
    "n_vallejo_rows": int(vallejo_view.shape[0]),
}

save_json(
    integration_metadata,
    artifact_path("integration", "integration_metadata.json"),
)

integration_metadata

{'base_artifact': 'data/poems_processed.csv',
 'unsupervised_artifact': 'artifacts/unsupervised/unsupervised_results.csv',
 'supervised_artifact': 'artifacts/supervised/supervised_predictions.csv',
 'integration_key_cols': ['title', 'poet', 'source', 'corpus_role'],
 'n_rows_integrated': 13858,
 'n_columns_integrated': 26,
 'n_clusters_km': 2,
 'n_topics_lda': 7,
 'n_vallejo_rows': 9}

## 5. Exportar artefactos de integración

In [7]:
df_to_save = df_integracion.copy()

list_like_cols = [
    "original_tags",
    "predicted_tags",
    "nearest_titles_cosine",
    "nearest_scores_cosine",
]

for col in list_like_cols:
    if col in df_to_save.columns:
        df_to_save[col] = df_to_save[col].apply(json_dumps_list)

save_csv(
    df_to_save,
    artifact_path("integration", "integrated_results.csv"),
)

save_csv(
    cluster_tag_matrix.reset_index(),
    artifact_path("integration", "cluster_tag_matrix.csv"),
)

save_csv(
    cluster_topic_matrix.reset_index(),
    artifact_path("integration", "cluster_topic_matrix.csv"),
)

save_csv(
    cluster_poet_matrix.reset_index(),
    artifact_path("integration", "cluster_poet_matrix.csv"),
)

save_csv(
    cluster_source_matrix.reset_index(),
    artifact_path("integration", "cluster_source_matrix.csv"),
)

save_csv(
    poet_tag_matrix.reset_index(),
    artifact_path("integration", "poet_tag_matrix.csv"),
)

save_csv(
    top_tags_cluster,
    artifact_path("integration", "top_tags_by_cluster.csv"),
)

vallejo_to_save = vallejo_view.copy()

for col in list_like_cols:
    if col in vallejo_to_save.columns:
        vallejo_to_save[col] = vallejo_to_save[col].apply(json_dumps_list)

save_csv(
    vallejo_to_save,
    artifact_path("integration", "vallejo_view.csv"),
)



print("Artefactos de integración guardados en artifacts/integration/")

Artefactos de integración guardados en artifacts/integration/
